# Walk Through L1 and L2

## 1) Objective
We learned in the lecture that two common regularization methods are L1 (aka "LASSO") and L2 (aka "RIDGE"), where L1 encourages sparcity and reduces overfitting and L2 penalizes large magnitudes of the coefficients $\beta$.<br>
In this example, we want to fit a logistic model to the diabetes data set we know from the Lab Course Module 6 and compare the result to the best result we obtain by implementing regularisation. 

## 2) Calling Libaries

For visualizations, for scanning the model for the best L1 and L2 parameters (Lagrangian Multiplier) and for bootstrapping (see later) we will use pre-defined custom functions which are stored in a file called ```Auxiliary.py```. This file also calls all the standard libraries we are going to need. Therefore, we load ```Auxiliary.py``` and check its methods:

In [ ]:
import sys  
sys.path.insert(1, '../10 Codes')

from Auxiliary import *

In [ ]:
help('Auxiliary')

<br>

## 3) Loading, Splitting and Scaling the Data

Next, we load the data set, separate ```X``` and ```Y```, scale the data accordingly and split the set into training and test set.

In [ ]:
location = FindMyFile('Diabetes2.csv')

In [ ]:
Data = pd.read_csv(location, sep = ';')
Data.head()

In [ ]:
X    = Data.drop(['label'], axis = 1)
Y    = Data['label']

Since we want to compare the test accuarcy from the standard model to the test accuracy we obtain from the best regularization parameters, we are aiming on a larger test set so that the results are more reliable. Alternatively, we could also run a k-fold cross-validation test.<br>
There are eight fit parameters, i. e. nine coefficients in total if we include the intercept. The data set contains almost 800 data points, i.e. even if we split training and test data in, say 60/40, the number of data points is more than an order of magnitude larger than the number of coefficients.

In [ ]:
X_Train, X_Test, Y_Train, Y_Test = train_test_split(X, Y, test_size = 0.40, random_state = 1)

Scaling the data:

In [ ]:
scaler       = StandardScaler()

XS_Train     = scaler.fit_transform(X_Train)
XS_Test      = scaler.transform(X_Test)

<br>

## 4) Standard Logistic Model

We first run the standard logistic model. The cross entropy plots, the accuracy and the confusion matrix will be the benchmark in order to compare it to the models with regularization.

In [ ]:
#adding the intercept to X and creating dummy variables for Y:
XS_Train_add = sm.add_constant(pd.DataFrame(XS_Train, columns = X.columns))
Ydumm        = pd.get_dummies(Y_Train).reset_index(drop=True)

In [ ]:
#the fit:
my_model = sm.GLM(Ydumm, XS_Train_add, family = sm.families.Binomial())
result   = my_model.fit()
result.summary()

We see that some of the coefficients (like e. g. for *SkinThickness*) have a large p-value and that their values are consitent with zero. Nevertheless, we evaluate the fit quality by calculating the **accuracy**:

In [ ]:
Probs  = result.predict(sm.add_constant(XS_Test))
ProbsR = np.round(Probs)
Ypred  = ['diabetes' if p== 1 else 'healthy'for p in ProbsR]

In [ ]:
acc = 100*sum(Ypred == Y_Test)/len(Y_Test)
print(f"Accuracy is {acc: .2f}%")

Next, we plot the **confusion matrix**:

In [ ]:
plot_confusion(Y_Test, Ypred,['diabetes', 'healthy'])

and create the **cross-entropy** plot

In [ ]:
Pmatrix = np.vstack((Probs, 1 - Probs)).transpose()
plot_entropy(Pmatrix, [1 if y== 'diabetes' else 0 for y in  Y_Test], ['diabetes', 'healthy'], [1, 0])

As we can see, the model is very confident in voting for *"healthy"*, but essentialy just guesses when voting for *"diabetes"*. Note also the inbalance of the different classes.

<br>

## 5) Regularization

In order to run a fit with regularization we need to run ```my_model.fit_regularized``` once we initialized ```my_model```. The standard method is ```'elastic_net'``` which provides a convenient way to balance both regularization terms. Let $\alpha_1$ and $\alpha_2$ be the regularization coefficients, then the Lagrangian $\mathcal{L}$ reads<br>
<br>
$\mathcal{L} = \mathcal{\bar{L}} + \alpha_1 \left(\frac{1-\alpha_2}{2}\,||\beta||^2_2 + \alpha_2\,||\beta||^1_1\right)$<br>
<br>
where $\mathcal{\bar{L}}$ is the Lagrangian/Loss without regularization, i. e. $\mathcal{\bar{L}} = ||Y-X\beta||^2_2$.<br>
As defined by the elastic net equation, setting $\alpha_2 = 1$ equals L1 regularization and $\alpha_2 = 0$ equals L2 regularization. Therefore, the actual Lagrangian multiplier are $\lambda_1 = \alpha_1\alpha_2$ and  $\lambda_2 = \alpha_1\frac{1-\alpha_2}{2}$.<br>

<br>

**5.1) Test Fit**

Next, we pick some values for $\alpha_1$ and $\alpha_2$ and see, if the result improves.  

In [ ]:
alpha1 = 0.1
alpha2 = 0.1

result_reg = my_model.fit_regularized(method = 'elastic_net', alpha = alpha1, L1_wt = alpha2)

Probs  = result_reg.predict(sm.add_constant(XS_Test))
ProbsR = np.round(Probs)
Ypred  = ['diabetes' if p== 1 else 'healthy'for p in ProbsR]

acc = 100*sum(Ypred == Y_Test)/len(Y_Test)
print(f"Accuracy is {acc: .2f}%")

We get pretty much the same value for the accuracy. 

<br>

**5.2) Systematic Scan for $\alpha_1$ and $\alpha_2$**

Since linear models are fast, we can scan for reasonable values for $\alpha_1$ and $\alpha_2$, calculate the fit and validation accuracy and pick the best combination for $\alpha_1$ and $\alpha_2$. For that purpose we run the function ```scan_regularization.py```. Note: the code will run for a few minutes. 

In [ ]:
my_model_best, BestEvalAcc, BestAlpha1, BestAlpha2 = scan_regularization(my_model, XS_Test, XS_Train, Y_Test, Y_Train)

In [ ]:
print(f" best accuracy: {100*BestEvalAcc: .2f}%\n alpha_1 = {BestAlpha1}\n alpha_2 = {BestAlpha2}")

The accuracy improved from 75.6% to almost 78%. That does not sound like a lot but seems to be pretty close to the theoretical limit since the two cluster are not that well separated.<br>
Next, we create the **cross-entropy** plot:

In [ ]:
Probs   = my_model_best.predict(sm.add_constant(XS_Test))
Pmatrix = np.vstack((Probs, 1 - Probs)).transpose()

plot_entropy(Pmatrix, [1 if y== 'diabetes' else 0 for y in  Y_Test], ['diabetes', 'healthy'], [1, 0])

Again, the result is not perfect, but the distribution for "diabetes" improved siginificantly. Therefore, the model generates fewer false negatives, which we can be seen in the **confusion matrix**:

In [ ]:
ProbsR     = np.round(Probs)
Ypred_best = ['diabetes' if p== 1 else 'healthy'for p in ProbsR]
plot_confusion(Y_Test, Ypred_best,['diabetes', 'healthy'])

<br>

<br>

**5.3) Bootstrapping for $\beta$**

One downside of regularization is that we can't calculate the confidence intervalls for the coefficients $\beta$ in the same straight forward way as before. A simple and relative robust method is bootstrapping (see Module 3) the dataset and calculating the confidence intervalls for the coefficients $\beta$ via percentiles from the bootstrap results.<br>
Note, that we can run $Nboot = 1000$ fits in a reasonable time because linear models are fast. For that purpose we call ```boot_strap_coeff.py```. The code will run for a few Minutes.

In [ ]:
lower, upper = boot_strap_coeff(np.array(XS_Train_add), Ydumm, best_a1 = BestAlpha1, best_a2 = BestAlpha1, Nboot = 1000)

In [ ]:
Coefbest = np.array(my_model_best.params)
CoefAll  = np.vstack((lower, Coefbest, upper)).transpose()
CoefDf   = pd.DataFrame(CoefAll, columns = ['conf 0.025', 'best', 'conf 0.975'], index = ['const'] + list(X.columns))

In [ ]:
#coefficients for best L1 and L2
CoefDf

In [ ]:
#coefficients without regularization (standard model)
result.summary2()

As we can see, the best regularization model returns coefficients which tend to be smaller in magintude compared to the coefficients calculated via the standard model. The best regularization model is also sparse. Especially those coefficients which had a large p-value in the standard model, are consistent to zero in the best regularization model within their 2$\sigma$ percentiles.

<br>

**5.4) The Sparse Model**

Encouraged by the above results, we revisit the original model, but this time without those features who seem to be not significant according to the best regularization model. The so called *"sparse model"* should perform as good as the original model.

In [ ]:
#for convenience, we turn the np.array into a df
XS_Test = pd.DataFrame(XS_Test, columns = X.columns)

In [ ]:
XS_Test.head()

Next, we exclude the features which turned out to be not relevant.

In [ ]:
XS_Test_sparse = XS_Test[['Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']]
XS_Test_sparse.head()

In [ ]:
XS_Train_sparse = XS_Train_add[['const', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']]
XS_Train_sparse.head()

We run the sparse model:

In [ ]:
my_model_sparse = sm.GLM(Ydumm, XS_Train_sparse, family = sm.families.Binomial())
result_sparse   = my_model_sparse.fit()
result_sparse.summary()

In [ ]:
Probs  = result_sparse.predict(sm.add_constant(XS_Test_sparse))
ProbsR = np.round(Probs)
Ypred  = ['diabetes' if p== 1 else 'healthy'for p in ProbsR]

In [ ]:
acc = 100*sum(Ypred == Y_Test)/len(Y_Test)
print(f"Accuracy is {acc: .2f}%")

The sparse model performs even **sligthly better** - and this time with **fewer coefficients!** One could still argue that *"DiabetesPedigreeFunction"* might not be significant too.<br>
We also check the confusion matrix and the cross-entropy plot.

In [ ]:
plot_confusion(Y_Test, Ypred,['diabetes', 'healthy'])

In [ ]:
Pmatrix = np.vstack((Probs, 1 - Probs)).transpose()
plot_entropy(Pmatrix, [1 if y== 'diabetes' else 0 for y in  Y_Test], ['diabetes', 'healthy'], [1, 0])

Finally, we fit the sparse model using the best regularization parameters that we found earlier in order to get the **final model**:

In [ ]:
alpha1 = BestAlpha1
alpha2 = BestAlpha2

result_reg_sparse = my_model_sparse.fit_regularized(method = 'elastic_net', alpha = alpha1, L1_wt = alpha2)

Probs  = result_reg_sparse.predict(sm.add_constant(XS_Test_sparse))
ProbsR = np.round(Probs)
Ypred  = ['diabetes' if p== 1 else 'healthy'for p in ProbsR]

acc = 100*sum(Ypred == Y_Test)/len(Y_Test)
print(f"Accuracy is {acc: .2f}%")

which again results in the same accuray, but with fewer parameter.

In [ ]:
plot_confusion(Y_Test, Ypred,['diabetes', 'healthy'])

In [ ]:
Pmatrix = np.vstack((Probs, 1 - Probs)).transpose()
plot_entropy(Pmatrix, [1 if y== 'diabetes' else 0 for y in  Y_Test], ['diabetes', 'healthy'], [1, 0])

Finally, we bootstrap the model in order to estimate the confidence intervals of the coefficients.

In [ ]:
lower, upper = boot_strap_coeff(np.array(XS_Train_sparse), Ydumm, best_a1 = BestAlpha1, best_a2 = BestAlpha1, Nboot = 1000)

In [ ]:
Coefbest = np.array(result_reg_sparse.params)
CoefAll  = np.vstack((lower, Coefbest, upper)).transpose()
CoefDf   = pd.DataFrame(CoefAll, columns = ['conf 0.025', 'best', 'conf 0.975'], index = list(XS_Train_sparse.columns))

In [ ]:
CoefDf

Which is out final model. We reduced the number for features from **nine to six** and thereby **increased the accuracy**, **improved cross-entropy** and even reduced the size of the error bars of the coefficients. 